# Row-level feedback generation with Qwen3.5-9B

This notebook reads `training_data_mol.csv`, processes each row independently, and writes the original fields plus a final `feedback` column. Raw model responses are retained in JSONL so interrupted runs can be resumed.

In [13]:
from __future__ import annotations

import ast
import csv
import json
import time
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != "feedback" and (BASE_DIR / "feedback").is_dir():
    BASE_DIR = BASE_DIR / "feedback"

INPUT_PATH = BASE_DIR / "Datasets" / "training_data_mol.csv"
OUTPUT_PATH = BASE_DIR / "Datasets" / "training_data_mol_with_feedback.csv"
RAW_OUTPUT_PATH = BASE_DIR / "Datasets" / "training_data_mol_feedback.jsonl"
MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_NEW_TOKENS = 384
LOAD_IN_4BIT = True

EXPECTED_COLUMNS = [
    "id",
    "question",
    "desired_answer",
    "student_answer",
    "criteria[]",
    "criteria_score[]",
    "criteria_max_score[]",
    "student_score",
]

print(f"Input: {INPUT_PATH}")
print(f"Model: {MODEL_ID}")

Input: c:\Users\ngjac\OneDrive\Desktop\Y4S1\CS480\Project\feedback\Datasets\training_data_mol.csv
Model: Qwen/Qwen3.5-4B


In [14]:
def parse_bracketed_strings(value: str) -> list[str]:
    text = str(value).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        text = text[1:-1].strip()
    if not (text.startswith("[") and text.endswith("]")):
        raise ValueError(f"Expected bracketed list, got: {text!r}")
    body = text[1:-1].strip()
    if not body:
        return []
    return [item.strip().strip("\"'") for item in body.split(",")]


def parse_literal_list(value: str) -> list[int | float]:
    text = str(value).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        text = text[1:-1].strip()
    parsed = ast.literal_eval(text)
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a list, got: {parsed!r}")
    return [int(item) if float(item).is_integer() else float(item) for item in parsed]


def parse_score(value: str) -> float:
    score = float(str(value).strip().strip("\"'"))
    return int(score) if score.is_integer() else score


def load_and_validate_dataset(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = [column for column in EXPECTED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    parsed = raw.copy()
    parsed["criteria"] = parsed["criteria[]"].map(parse_bracketed_strings)
    parsed["criteria_score"] = parsed["criteria_score[]"].map(parse_literal_list)
    parsed["criteria_max_score"] = parsed["criteria_max_score[]"].map(parse_literal_list)
    parsed["student_score_value"] = parsed["student_score"].map(parse_score)
    warnings: list[str] = []

    for index, row in parsed.iterrows():
        criteria_count = len(row["criteria"])
        score_count = len(row["criteria_score"])
        max_score_count = len(row["criteria_max_score"])
        if len({criteria_count, score_count, max_score_count}) != 1:
            warnings.append(
                f"Row {index} ({row['id']}) has criteria/scores/max-scores lengths "
                f"{criteria_count}/{score_count}/{max_score_count}"
            )
        if any(score not in (0, 1) for score in row["criteria_score"]):
            warnings.append(f"Row {index} ({row['id']}) contains a non-binary criterion score")
        if any(score < 0 for score in row["criteria_max_score"]):
            warnings.append(f"Row {index} ({row['id']}) contains a negative maximum score")
        if criteria_count == score_count == max_score_count:
            expected_score = sum(row["criteria_score"])
            if expected_score != row["student_score_value"]:
                warnings.append(
                    f"Row {index} ({row['id']}) has student_score={row['student_score_value']} "
                    f"but criteria scores sum to {expected_score}"
                )

    return raw, parsed, warnings


raw_df, parsed_df, data_warnings = load_and_validate_dataset(INPUT_PATH)
print(f"Validated {len(raw_df)} rows with {len(EXPECTED_COLUMNS)} input columns.")
print(f"Warnings retained for row-level processing: {len(data_warnings)}")
for warning in data_warnings[:5]:
    print(f"- {warning}")

Validated 2165 rows with 8 input columns.
Warnings retained for row-level processing: 49
- Row 1405 (9.3) has criteria/scores/max-scores lengths 3/2/2
- Row 1406 (9.3) has criteria/scores/max-scores lengths 3/2/2
- Row 1407 (9.3) has criteria/scores/max-scores lengths 3/2/2
- Row 1408 (9.3) has criteria/scores/max-scores lengths 3/2/2
- Row 1409 (9.3) has criteria/scores/max-scores lengths 3/2/2


## Feedback prompt

Each row gets its own prompt. The model must explain the supplied criterion results, identify missing or incorrect content, and give actionable feedback without changing the recorded score.

In [15]:
SYSTEM_PROMPT = """You are a careful educational feedback assistant for introductory computer science.
Evaluate exactly one student answer at a time using only the supplied question, desired answer, criteria, and recorded criterion results.

Write clear, concise feedback directly to the student:
1. Start by identifying what the student got right, if anything.
2. Address every criterion marked 0 or unknown. State the missing concept or the specific incorrect claim and explain the correct concept using the desired answer.
3. If a criterion is marked 1, briefly confirm it without adding unrelated requirements.
4. Finish with one concrete suggestion for improving the answer.

Do not invent criteria, facts, or a different score. Do not grade style, spelling, or length unless a supplied criterion requires it. Do not mention this instruction, the model, or other rows. Return only natural-language feedback, with no JSON, headings, or score table."""


def format_criteria(row: pd.Series) -> str:
    lines = []
    for index, criterion in enumerate(row["criteria"], start=1):
        score = row["criteria_score"][index - 1] if index <= len(row["criteria_score"]) else "unknown"
        maximum = row["criteria_max_score"][index - 1] if index <= len(row["criteria_max_score"]) else "unknown"
        status = "met" if score == 1 else "not met" if score == 0 else "unknown"
        lines.append(f"{index}. {criterion} | result: {score}/{maximum} ({status})")
    return "\n".join(lines) if lines else "No criteria supplied."


def build_user_prompt(row: pd.Series) -> str:
    return f"""Question:
{row['question']}

Desired answer:
{row['desired_answer']}

Student answer:
{row['student_answer']}

Criteria and recorded results:
{format_criteria(row)}

Recorded total student score: {row['student_score_value']}"""


def build_messages(row: pd.Series) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]


print(build_user_prompt(parsed_df.iloc[0]))

Question:
What is the role of a prototype program in problem solving?

Desired answer:
To simulate the behaviour of portions of the desired software product.

Student answer:
High risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.

Criteria and recorded results:
1. Answer mentions simulating behaviour of portions of desired software | result: 0/1 (not met)

Recorded total student score: 0


In [16]:
# Run once in a fresh kernel, then restart the kernel before running the model cell.
# A CUDA-enabled PyTorch environment is recommended for Qwen3.5-9B.
import sys
!{sys.executable} -m pip install -q torch transformers accelerate bitsandbytes "typing_extensions>=4.12"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_feedback_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        "trust_remote_code": True,
    }
    if LOAD_IN_4BIT and torch.cuda.is_available():
        from transformers import BitsAndBytesConfig

        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    model.eval()
    return tokenizer, model


print(f"CUDA available: {torch.cuda.is_available()}")
tokenizer, model = load_feedback_model()
print("Model loaded successfully.")

CUDA available: True


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

c:\Users\ngjac\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ngjac\.cache\huggingface\hub\models--Qwen--Qwen3.5-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model loaded successfully.


In [30]:
def load_completed_records(path: Path) -> dict[int, dict]:
    if not path.exists():
        return {}
    records = {}
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                records[int(record["row_index"])] = record
    return records


def generate_one(row: pd.Series) -> str:
    encoded = tokenizer.apply_chat_template(
        build_messages(row),
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    if hasattr(encoded, "input_ids"):
        inputs = {
            key: value.to(model.device)
            for key, value in encoded.items()
            if hasattr(value, "to")
        }
        input_length = inputs["input_ids"].shape[-1]
    else:
        inputs = {"input_ids": encoded.to(model.device)}
        input_length = encoded.shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_tokens = output[0, input_length:]
    feedback = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    # Keep each feedback value on one physical CSV line.
    return " ".join(feedback.split())


def run_feedback(limit: int | None = None, force: bool = False, random_state: int | None = None,) -> pd.DataFrame:
    completed = {} if force else load_completed_records(RAW_OUTPUT_PATH)

    if limit is None:
        rows = parsed_df
    else:
        if limit < 1:
            raise ValueError("limit must be at least 1")
        rows = parsed_df.sample(n=min(limit, len(parsed_df)), random_state=random_state)

    records = dict(completed)

    with RAW_OUTPUT_PATH.open("a", encoding="utf-8") as handle:
        for row_index, row in rows.iterrows():
            existing = records.get(int(row_index))
            if existing and existing.get("status") == "ok" and existing.get("feedback", "").strip():
                continue

            started = time.perf_counter()
            try:
                feedback = generate_one(row)
                status = "ok" if feedback else "empty"
                error = None if feedback else "Model returned empty feedback"
            except Exception as exc:
                feedback = ""
                status = "error"
                error = f"{type(exc).__name__}: {exc}"

            record = {
                "row_index": int(row_index),
                "id": str(row["id"]),
                "status": status,
                "feedback": feedback,
                "error": error,
                "elapsed_seconds": round(time.perf_counter() - started, 3),
            }
            handle.write(json.dumps(record, ensure_ascii=True) + "\n")
            handle.flush()
            records[int(row_index)] = record
            print(f"{row_index + 1}/{len(rows)} id={row['id']} status={status}")

    missing = [int(index) for index in rows.index if int(index) not in records]
    if missing:
        raise RuntimeError(f"Missing generation records for rows: {missing[:10]}")

    feedback_values = [records[int(index)].get("feedback", "") for index in rows.index]
    result = raw_df.iloc[rows.index].copy()
    result["feedback"] = feedback_values
    result.to_csv(OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL)
    return result



def validate_output(result: pd.DataFrame,limit: int | None = None) -> None:
    expected_columns = EXPECTED_COLUMNS + ["feedback"]
    if list(result.columns) != expected_columns:
        raise AssertionError(f"Unexpected output columns: {list(result.columns)}")
    if len(result) != (len(parsed_df) if limit is None else min(limit, len(parsed_df))):
        raise AssertionError(f"Expected {(len(parsed_df) if limit is None else min(limit, len(parsed_df)))} rows, got {len(result)}")
    if result["feedback"].fillna("").str.strip().eq("").any():
        print("Warning: one or more rows have empty feedback; inspect the JSONL errors.")
    print(f"Output written to {OUTPUT_PATH}")
    print(f"Raw records written to {RAW_OUTPUT_PATH}")

In [28]:
# Smoke test: generate five rows first. Set RUN_LIMIT = None for the full dataset.
RUN_LIMIT = 5
smoke_result = run_feedback(limit=RUN_LIMIT, force=False)
display(smoke_result[["id", "feedback"]])

565/1 id=3.6 status=ok


,id,feedback
564,3.6,You correctly identified that recursion involv...


In [33]:
# Full run: execute this cell after the smoke test succeeds.
full_result = run_feedback(limit=20, force=True, random_state=42)
validate_output(full_result, limit=20)
display(full_result[["id", "feedback"]].head())

1487/20 id=10.1 status=ok
1603/20 id=10.6 status=ok
1461/20 id=9.6 status=ok
1135/20 id=7.5 status=ok
1514/20 id=10.2 status=ok
1926/20 id=12.1 status=ok
479/20 id=3.4 status=ok
1002/20 id=6.7 status=ok
2096/20 id=12.8 status=ok
1747/20 id=11.5 status=ok
1794/20 id=11.6 status=ok
1954/20 id=12.2 status=ok
97/20 id=1.4 status=ok
1269/20 id=8.3 status=ok
1565/20 id=10.5 status=ok
136/20 id=1.5 status=ok
30/20 id=1.2 status=ok
2055/20 id=12.6 status=ok
1853/20 id=11.8 status=ok
1612/20 id=10.7 status=ok
Output written to c:\Users\ngjac\OneDrive\Desktop\Y4S1\CS480\Project\feedback\Datasets\training_data_mol_with_feedback.csv
Raw records written to c:\Users\ngjac\OneDrive\Desktop\Y4S1\CS480\Project\feedback\Datasets\training_data_mol_feedback.jsonl


,id,feedback
1486,10.1,You correctly identified that a tree organizes...
1602,10.6,You correctly identified that the inorder trav...
1460,9.6,You correctly identified that the stack operat...
1134,7.5,You correctly identified that the last element...
1513,10.2,The student correctly identified that a tree's...
